# Chapter 3: Active Space Selection

## What you'll learn

- Why the choice of active space directly determines qubit cost and why there's no single right answer
- The full menu of QDK active space selectors: `qdk_valence`, `pyscf_avas`, `qdk_occupation`, `qdk_autocas`, and `qdk_autocas_eos`
- When each method is appropriate and how to interpret what they return
- How to compare active spaces across methods by their qubit footprint

## The fundamental tradeoff

Active space selection determines how expensive your quantum calculation will be. Typically, every spatial orbital you include costs **two qubits** (one per spin). A 10-orbital active space needs 20 qubits; a 14-orbital space needs 28. This assumes the Jordan Wigner mapping, which you will learn more about in the next chapter! The exponential scaling of exact diagonalization means this difference is not cosmetic.

The goal is to find the **smallest active space that still captures the strongly correlated physics**. In traditional quantum chemistry, this relies on the chemist's expertise: knowledge of which orbitals are chemically relevant — bonding, antibonding, lone pairs — informed by the system's symmetry and past experience with similar molecules. This works well for well-understood systems but can miss correlations in unfamiliar territory or require significant trial and error.

The QDK provides a spectrum of selectors that reduce or eliminate this reliance on prior knowledge — ranging from methods that encode chemical knowledge explicitly to methods that derive the active space automatically from computed properties of the wavefunction:

| Selector | Basis for selection | Requires |
|---|---|---|
| `qdk_valence` | Valence shell heuristic | HF wavefunction |
| `pyscf_avas` | AO-to-MO projection overlap | HF wavefunction |
| `qdk_occupation` | Natural orbital occupation numbers | Wavefunction with 1-RDM |
| `qdk_autocas` | Single-orbital entanglement entropy | Wavefunction with 1- and 2-RDM |
| `qdk_autocas_eos` | Entropy of substitution (EOS) | Wavefunction with 1- and 2-RDM |

In [ ]:
# Setup: SCF, valence space, and MP2 localization
from pathlib import Path

import qdk_chemistry.plugins.pyscf

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)

# SCF
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
print(f"HF energy: {E_hf:.6f} Hartree")

# Valence space
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
valence_selector = create("active_space_selector", "qdk_valence",
                          num_active_electrons=num_val_e, num_active_orbitals=num_val_o)
wfn_valence = valence_selector.run(wfn_hf)
valence_indices = wfn_valence.get_orbitals().get_active_space_indices()

# MP2 natural orbital localization
localizer_mp2 = create("orbital_localizer", "qdk_mp2_natural_orbitals")
wfn_mp2 = localizer_mp2.run(wfn_valence, *valence_indices)
print(f"Valence space: {num_val_e} electrons, {num_val_o} orbitals")
print(f"Starting active space indices: {valence_indices[0]}")


## Part 1: What selectors are available?

`available("active_space_selector")` lists every registered selector. `print_settings()` shows each one's configurable parameters. Run the cell to see them all.

In [ ]:
# List available active space selectors
selectors = available("active_space_selector")
print(f"Available active space selectors: {selectors}")

print()
for name in selectors:
    print(f"\n--- {name} ---")
    print_settings("active_space_selector", name)


## Part 2: qdk_valence — manual tuning

`qdk_valence` is the simplest selector: it takes explicit electron and orbital counts and carves out that subspace from the SCF wavefunction. `compute_valence_space_parameters()` infers sensible defaults, but those are just a starting point.

For stretched N₂ with cc-pvdz, the valence heuristic returns 10 electrons in 8 orbitals. You can shrink to (6e, 6o) to focus on the σ-system only, or expand to (10e, 10o) to include more virtuals. Each choice has a different qubit cost.

Let's run three `qdk_valence` configurations and compare their qubit footprints.

In [ ]:
# Three manual active space choices for stretched N₂
choices = [(6, 6), (num_val_e, num_val_o), (10, 10)]
labels  = ["minimal (6e,6o)", "valence default (10e,8o)", "expanded (10e,10o)"]

valence_wfns = {}
for (ne, no), label in zip(choices, labels):
    sel = create("active_space_selector", "qdk_valence",
                 num_active_electrons=ne, num_active_orbitals=no)
    wfn = sel.run(wfn_hf)
    valence_wfns[label] = wfn
    active_o = wfn.get_orbitals().get_active_space_indices()[0]
    n_qubits = 2 * len(active_o)
    print(f"{label}: {len(active_o)} active orbitals → {n_qubits} qubits")
    print(f"  indices: {list(active_o)}")


## Part 3: pyscf_avas — AO-overlap selection

`pyscf_avas` (<a href="https://pubs.acs.org/doi/10.1021/acs.jctc.7b00128" target="_blank">Sayfutyarova et al., 2017</a>) (Atomic Valence Active Space) encodes prior chemical knowledge explicitly: you specify which atomic orbital character the active space should contain (e.g., `["N 2s", "N 2p"]`), and AVAS projects those onto the molecular orbital basis, retaining the MOs with the largest overlap weight.

This makes AVAS the right choice when you already know which bonds or lone pairs drive the chemistry. It translates that knowledge directly into a selection criterion. For N₂ dissociation, including both 2s and 2p captures the σ and π bonds. Like `qdk_valence`, AVAS works directly from the HF wavefunction and no multi-configuration step needed.

Fill in the missing AO label in the cell below and then answer the following question. 

In [ ]:
# AVAS: atomic-orbital-guided selection
# Specify which atomic orbital character to include — AVAS projects these AOs
# onto the MO basis and selects the orbitals with the largest overlap weight.
avas = create("active_space_selector", "pyscf_avas", ao_labels=["N 2s", "???"])  # fill in: add 2p character for pi bonds
wfn_avas = avas.run(wfn_hf)
avas_indices = wfn_avas.get_orbitals().get_active_space_indices()[0]
print(f"AVAS (2s+2p, sigma+pi): {len(avas_indices)} orbitals → {2*len(avas_indices)} qubits")
print(f"Selected orbital indices: {list(avas_indices)}")

# Unlike qdk_valence (heuristic), AVAS is orbital-character-guided: you specify the atomic
# orbital character that matters for the bonding you want to describe.
# For N₂ dissociation, 2s+2p captures both the sigma and pi bonds.
# AVAS uses the HF wavefunction directly (no SCI step needed).


## Exercise: What are the selected orbital indices you get after running the AVAS cell?

What are the selected orbital indices you get after running the AVAS cell? Paste the list you get in the output. 

### Expected Answer

['[2, 3, 4, 5, 6, 7, 8, 9]']


## Part 4: Entropy-based selection — qdk_autocas and qdk_autocas_eos

The autoCAS selectors (<a href="https://pubs.acs.org/doi/10.1021/acs.jctc.6b00156" target="_blank">Stein & Reiher, 2016</a>) are the most **automated** option: they derive the active space directly from the wavefunction's quantum information structure, requiring no prior knowledge of the molecule's chemistry. They compute **single-orbital entanglement entropies** from the 2-RDM of a multi-configuration wavefunction and orbitals with high entropy are strongly entangled with the rest of the system and belong in the active space; orbitals near zero can be frozen.

This requires a wavefunction with 1- and 2-RDM:
1. A classical Hamiltonian for the active space (via `hamiltonian_constructor`)
2. A multi-configuration wavefunction with 1-RDM and 2-RDM (via `multi_configuration_calculator`, here MACIS ASCI (<a href="https://aip.scitation.org/doi/10.1063/1.4955109" target="_blank">Tubman et al., 2016</a>))

`qdk_autocas_eos` adds an *entropy of substitution* step: it tests whether swapping each active orbital changes the entropy landscape, catching orbitals the standard criterion misses.

In the cells below, we build the Hamiltonian from `wfn_mp2`, run MACIS ASCI with RDMs, then apply both autoCAS methods. Compare what each selects.

In [ ]:
# Build Hamiltonian and run MACIS ASCI with RDMs
hamiltonian_constructor = create("hamiltonian_constructor")
loc_hamiltonian = hamiltonian_constructor.run(wfn_mp2.get_orbitals())
print("Hamiltonian summary:")
print(loc_hamiltonian.get_summary())

# Run MACIS ASCI with 1-RDM and 2-RDM
# The autoCAS selectors read orbital entanglement entropies from the RDMs.
num_alpha, num_beta = wfn_mp2.get_active_num_electrons()
print(f"\nActive electrons: \u03b1={num_alpha}, \u03b2={num_beta}")

macis = create("multi_configuration_calculator", "macis_asci",
               calculate_one_rdm=True,
               calculate_two_rdm=True)
macis.settings().set("core_selection_strategy", "fixed")  # required when starting from a single HF determinant

e_sci, wfn_sci = macis.run(loc_hamiltonian, num_alpha, num_beta)
print(f"\nMACIS ASCI energy: {e_sci:.6f} Hartree")
print(f"Number of determinants: {len(wfn_sci.get_active_determinants())}")

In [ ]:
# Apply standard autoCAS: entropy-based selection
autocas = create("active_space_selector", "qdk_autocas")
print("autoCAS settings:")
print_settings("active_space_selector", "qdk_autocas")

wfn_autocas = autocas.run(wfn_sci)
autocas_indices = wfn_autocas.get_orbitals().get_active_space_indices()[0]
print(f"\nautoCAS selected {len(autocas_indices)} orbitals: {list(autocas_indices)}")
print(f"Qubit cost: {2 * len(autocas_indices)}")


In [ ]:
# Apply autoCAS-EOS: entropy-of-substitution refinement
autocas_eos = create("active_space_selector", "qdk_autocas_eos")
print("autoCAS-EOS settings:")
print_settings("active_space_selector", "qdk_autocas_eos")

wfn_eos = autocas_eos.run(wfn_sci)
eos_indices = wfn_eos.get_orbitals().get_active_space_indices()[0]
print(f"\nautoCAS-EOS selected {len(eos_indices)} orbitals: {list(eos_indices)}")
print(f"Qubit cost: {2 * len(eos_indices)}")

# Compare the two entropy methods
print(f"\n--- Entropy method comparison ---")
print(f"autoCAS:     {list(autocas_indices)}")
print(f"autoCAS-EOS: {list(eos_indices)}")
in_eos_not_autocas = set(eos_indices) - set(autocas_indices)
if in_eos_not_autocas:
    print(f"EOS added orbital(s): {sorted(in_eos_not_autocas)} (captured by substitution entropy, missed by single-orbital entropy)")
else:
    print("Both methods agree on the active space.")


## Quiz: Which active space selector requires a multi-configuration wavefunction with ...

Which active space selector requires a multi-configuration wavefunction with 1-RDM and 2-RDM?

### Choices

A. qdk_valence
B. pyscf_avas
C. qdk_autocas
D. None of the above


## Part 5: qdk_occupation — occupation-based selection

`qdk_occupation` selects orbitals based on their **natural orbital occupation numbers** from the 1-RDM. In a strongly correlated system, orbitals with occupations near 1 (far from the fully-occupied value of 2 or fully-empty value of 0) are partially filled, and are a direct indicator of correlation.

This is complementary to entropy: entropy measures *entanglement* between an orbital and the rest; occupation numbers measure the orbital's *partial filling*. For stretched N₂, the σ-bonding and σ*-antibonding orbitals should both show occupation near 1. Like the autoCAS methods, it reads from the 1-RDM and so takes the post-HF wavefunction from Part 4.

Now, let's apply `qdk_occupation` to `wfn_sci`. Inspect the threshold setting, then compare to the autoCAS result.

In [ ]:
# Apply occupation-based selection
occ_selector = create("active_space_selector", "qdk_occupation")
print("Occupation selector settings:")
print_settings("active_space_selector", "qdk_occupation")

wfn_occ = occ_selector.run(wfn_sci)
occ_indices = wfn_occ.get_orbitals().get_active_space_indices()[0]
print(f"\nOccupation-based selection: {len(occ_indices)} orbitals: {list(occ_indices)}")
print(f"Qubit cost: {2 * len(occ_indices)}")


## Part 6: Cost vs. accuracy comparison

Each selector has made a different bet about which orbitals matter. Now compare them on two dimensions: orbital count and energy accuracy.

The first cell compares the number of active orbitals selected by each method — the direct output of active space selection, and the primary driver of downstream cost (Hamiltonian size, circuit depth, qubit count under any encoding). You will learn more about these downstream effects in the next chapter. The second runs exact diagonalization on the four selections that share the same MP2-localized orbital basis, and so their energies are directly comparable. This is the cost-vs-accuracy tradeoff: a smaller active space is cheaper but misses correlation energy.

In [ ]:
# Active orbital count: all selectors compared
results = [
    ("qdk_valence (minimal 6e,6o)",    valence_wfns["minimal (6e,6o)"].get_orbitals().get_active_space_indices()[0]),
    ("qdk_valence (default 10e,8o)",   valence_wfns["valence default (10e,8o)"].get_orbitals().get_active_space_indices()[0]),
    ("qdk_valence (expanded 10e,10o)", valence_wfns["expanded (10e,10o)"].get_orbitals().get_active_space_indices()[0]),
    ("pyscf_avas (N 2s+2p)",          avas_indices),
    ("qdk_autocas",                    autocas_indices),
    ("qdk_autocas_eos",                eos_indices),
    ("qdk_occupation",                 occ_indices),
]

print(f"{'Method':<35} {'Orbitals':>9} {'Qubits':>8} {'Indices'}")
print("-" * 75)
for name, indices in results:
    n = len(indices)
    print(f"{name:<35} {n:>9} {2*n:>8}   {list(indices)}")

In [ ]:
# Energy vs. cost: exact diagonalization across active space sizes
# wfn_mp2, wfn_autocas, wfn_eos, wfn_occ all use the same MP2-localized orbital basis
# so their energies are directly comparable — larger active space = lower (more accurate) energy
ham_constructor = create("hamiltonian_constructor")
qubit_mapper = create("qubit_mapper", "qdk")
solver = create("qubit_hamiltonian_solver", "qdk_sparse_matrix_solver")

energy_cases = [
    ("valence (10e,8o) \u2014 full",  wfn_mp2),
    ("qdk_autocas",                    wfn_autocas),
    ("qdk_autocas_eos",                wfn_eos),
    ("qdk_occupation",                 wfn_occ),
]

print(f"{'Method':<32} {'Qubits':>8} {'Energy (Hartree)':>20}")
print("-" * 64)
for name, wfn in energy_cases:
    orbitals = wfn.get_orbitals()
    n_orb = len(orbitals.get_active_space_indices()[0])
    ham = ham_constructor.run(orbitals)
    qubit_ham = qubit_mapper.run(ham)
    energy, _ = solver.run(qubit_ham)
    print(f"{name:<32} {2*n_orb:>8} {energy:>20.6f}")

# Note: qdk_occupation selected only 2 orbitals for N₂ — an extremely small active
# space. The solver returns E ≈ 0 because it finds the particle-number vacuum sector
# (zero active electrons), not the physical ground state. Very small active spaces can
# produce non-physical diagonalization results; always validate against a larger reference.

print(f"\n{'HF reference':<32} {'\u2014':>8} {E_hf:>20.6f}  (no correlation)")
print(f"{'MACIS ASCI (valence)':<32} {'\u2014':>8} {e_sci:>20.6f}  (multi-ref reference)")
print("\n\u2192 Larger active space \u2192 more qubits, lower energy")
print("\u2192 Carrying forward wfn_eos (autoCAS-EOS) into Chapter 4")


## Quiz: For stretched N₂, which selector tends to produce the most physically princip...

For stretched N₂, which selector tends to produce the most physically principled active space?

### Choices

A. qdk_valence with manually chosen electron/orbital counts
B. qdk_autocas using single orbital entanglement entropy
C. qdk_occupation using natural orbital occupation numbers 
D. qdk_autocas_eos using entropy of substitution 


## Summary

In this chapter you:
- Surveyed all five active space selectors and their settings
- Applied `qdk_valence` directly from the HF wavefunction with three manual configurations
- Applied `pyscf_avas` with AO labels (`"N 2s"`, `"N 2p"`) to select by atomic character from the HF wavefunction
- Built a MACIS ASCI post-HF wavefunction with 1- and 2-RDM, then ran entropy-based (`qdk_autocas`, `qdk_autocas_eos`) and occupation-based (`qdk_occupation`) selectors
- Compared all selections by active orbital count and energy accuracy for stretched N₂ with cc-pvdz

The autoCAS-EOS wavefunction (`wfn_eos`) is the recommended output to carry forward: it uses the most automated, knowledge-free selection criterion and is the input to Hamiltonian construction in Chapter 4.

**Key pattern:**
```python
# Build Hamiltonian → run CASCI with RDMs → autoCAS-EOS
hamiltonian_constructor = create("hamiltonian_constructor")
loc_hamiltonian = hamiltonian_constructor.run(wfn_mp2.get_orbitals())

macis = create("multi_configuration_calculator", "macis_asci",
               calculate_one_rdm=True, calculate_two_rdm=True)
macis.settings().set("core_selection_strategy", "fixed")
_, wfn_sci = macis.run(loc_hamiltonian, *wfn_mp2.get_active_num_electrons())

wfn_active = create("active_space_selector", "qdk_autocas_eos").run(wfn_sci)
```